In [2]:
from pathlib import Path
import shutil, zipfile

# ---------- Configuration ----------
ROOT = Path("..").resolve()                 # racine du repo (notebook dans notebooks/)
RAW  = ROOT / "data/raw"
OUT  = RAW / "vish_darija_tts" / "extracted"
META = OUT / "metadata"

# zip : cherché automatiquement dans data/raw (sinon écris le chemin à la main)
ZIP = next(RAW.rglob("*.zip"), None)
# ZIP = Path(r"C:\Users\chern\...\ton_fichier.zip")     # <- à décommenter si introuvable
assert ZIP is not None, "Zip introuvable : écris son chemin dans ZIP"
print("ROOT :", ROOT)
print("ZIP  :", ZIP)

# ---------- 1) déplacer l'ancienne extraction ----------
old = RAW / "extracted"
if old.exists() and not OUT.exists():
    OUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(old), str(OUT))
    print("Déplacé :", old, "->", OUT)

# ---------- 2) espace disque ----------
free = shutil.disk_usage(ROOT).free / 1e9
print(f"Libre : {free:.1f} Go")
assert free > 6, "Pas assez de place : libère de l'espace avant d'extraire"

# ---------- 3) extraire uniquement audio/clean_normalized ----------
with zipfile.ZipFile(ZIP) as z:
    todo = [i for i in z.infolist()
            if i.filename.startswith("audio/clean_normalized/")
            and not i.is_dir() and not (OUT / i.filename).exists()]
    print(len(todo), "fichiers à extraire")
    for k, i in enumerate(todo, 1):
        z.extract(i, OUT)
        if k % 500 == 0:
            print(f"{k}/{len(todo)}")

n_wav = len(list((OUT / "audio/clean_normalized").rglob("*.wav")))
print("WAV propres sur disque :", n_wav, "(attendu : 3400)")

ROOT : C:\Users\chern\Desktop\vishing-detection-1
ZIP  : C:\Users\chern\Desktop\vishing-detection-1\data\raw\VISH-DARIJA-TTS-v1.0.zip
Libre : 7.2 Go
3400 fichiers à extraire
500/3400
1000/3400
1500/3400
2000/3400
2500/3400
3000/3400
WAV propres sur disque : 3400 (attendu : 3400)


In [4]:
from pathlib import Path
import pandas as pd, soundfile as sf

ROOT = Path("..").resolve()

man = pd.read_csv(ROOT / "data/metadata/audio_manifest.csv")
print("Lignes du manifest :", len(man))
print("Colonnes :", list(man.columns))

# 1) tous les fichiers existent ?
missing = [p for p in man["audio_path"] if not (ROOT / p).exists()]
print("Fichiers manquants :", len(missing))            # attendu : 0
if missing:
    print("Exemple attendu :", missing[0])

# 2) en-têtes WAV cohérents avec le manifest (200 fichiers au hasard)
bad = 0
for r in man.sample(200, random_state=42).itertuples():
    p = ROOT / r.audio_path
    if not p.exists():
        continue
    info = sf.info(p)
    if info.samplerate != 24000 or info.channels != 1 or abs(info.duration - r.duration_sec) > 0.05:
        bad += 1
        print("écart :", r.sample_id, info.samplerate, info.channels,
              round(info.duration, 2), r.duration_sec)
print("Écarts d'en-tête sur 200 fichiers :", bad)      # attendu : 0

# 3) répartition rapide
print(man.groupby(["split", "label"]).size().unstack())

Lignes du manifest : 3400
Colonnes : ['sample_id', 'audio_path', 'label', 'label_original', 'language', 'source_dataset', 'speaker_id', 'scenario_id', 'is_synthetic', 'duration_sec', 'sample_rate', 'split', 'license']
Fichiers manquants : 0
Écarts d'en-tête sur 200 fichiers : 0
label          0     1
split                 
test         255   255
train       1190  1190
validation   255   255
